<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 01 — Ingestão RavenStack (challenge 001)

Lê os 5 CSVs de `dados/` e grava tabelas Delta. Colunas **somente as do CSV**.

### Controle de Alteração de Versões

| versão | data | autor | e-mail | alterações |
| --- | --- | --- | --- | --- |
| 1.2 | 21-SET-2026 | Adriano Aguiar | — | Troca para os 5 CSVs RavenStack do challenge 001 |

### Contrato dos CSVs (contado no disco 2026-09-21)

| Arquivo | Linhas | Colunas |
| --- | ---: | --- |
| `ravenstack_accounts.csv` | 500 | `account_id`, `account_name`, `industry`, `country`, `signup_date`, `referral_source`, `plan_tier`, `seats`, `is_trial`, `churn_flag` |
| `ravenstack_subscriptions.csv` | 5000 | `subscription_id`, `account_id`, `start_date`, `end_date`, `plan_tier`, `seats`, `mrr_amount`, `arr_amount`, `is_trial`, `upgrade_flag`, `downgrade_flag`, `churn_flag`, `billing_frequency`, `auto_renew_flag` |
| `ravenstack_feature_usage.csv` | 25000 | `usage_id`, `subscription_id`, `usage_date`, `feature_name`, `usage_count`, `usage_duration_secs`, `error_count`, `is_beta_feature` |
| `ravenstack_support_tickets.csv` | 2000 | `ticket_id`, `account_id`, `submitted_at`, `closed_at`, `resolution_time_hours`, `priority`, `first_response_time_minutes`, `satisfaction_score`, `escalation_flag` |
| `ravenstack_churn_events.csv` | 600 | `churn_event_id`, `account_id`, `churn_date`, `reason_code`, `refund_amount_usd`, `preceding_upgrade_flag`, `preceding_downgrade_flag`, `is_reactivation`, `feedback_text` |

Joins: `account_id`, `subscription_id`.

Label canônico de churn = linha em `ravenstack_churn_events`. `accounts.churn_flag` **diverge** do evento — não usar o flag como verdade.

### Como executar no Databricks

1. Importe a pasta `notebook-data-001-churn` para o Workspace (incluindo `dados/`).
2. Widgets: `catalog_name` (default `workspace`), `schema_name` (`ravenstack`), `data_path` (vazio = auto).
3. `data_path` se o auto-detect falhar: `/Workspace/Users/<voce>/notebook-data-001-churn/dados` ou um Volume `/Volumes/<cat>/<sch>/<vol>/dados`.
4. Cluster ou serverless → **Run All**.
5. Lab 02 e Lab 04 leem as tabelas `ravenstack_*` deste schema.

In [ ]:
from pathlib import Path

EXPECTED = {
    "ravenstack_accounts": {"rows": 500, "columns": [
        "account_id", "account_name", "industry", "country", "signup_date",
        "referral_source", "plan_tier", "seats", "is_trial", "churn_flag",
    ]},
    "ravenstack_subscriptions": {"rows": 5000, "columns": [
        "subscription_id", "account_id", "start_date", "end_date", "plan_tier",
        "seats", "mrr_amount", "arr_amount", "is_trial", "upgrade_flag",
        "downgrade_flag", "churn_flag", "billing_frequency", "auto_renew_flag",
    ]},
    "ravenstack_feature_usage": {"rows": 25000, "columns": [
        "usage_id", "subscription_id", "usage_date", "feature_name",
        "usage_count", "usage_duration_secs", "error_count", "is_beta_feature",
    ]},
    "ravenstack_support_tickets": {"rows": 2000, "columns": [
        "ticket_id", "account_id", "submitted_at", "closed_at",
        "resolution_time_hours", "priority", "first_response_time_minutes",
        "satisfaction_score", "escalation_flag",
    ]},
    "ravenstack_churn_events": {"rows": 600, "columns": [
        "churn_event_id", "account_id", "churn_date", "reason_code",
        "refund_amount_usd", "preceding_upgrade_flag", "preceding_downgrade_flag",
        "is_reactivation", "feedback_text",
    ]},
}

BOOL_COLUMNS = {
    "is_trial", "churn_flag", "upgrade_flag", "downgrade_flag",
    "auto_renew_flag", "is_beta_feature", "escalation_flag",
    "preceding_upgrade_flag", "preceding_downgrade_flag", "is_reactivation",
}
DATE_COLUMNS = {"signup_date", "start_date", "end_date", "usage_date", "churn_date"}
TIMESTAMP_COLUMNS = {"submitted_at", "closed_at"}


def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def resolve_data_path(explicit: str) -> str:
    marker = "ravenstack_accounts.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-data-001-churn" in parts:
            idx = parts.index("notebook-data-001-churn")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
            candidates.append(Path("/") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend(
        [
            here / "dados",
            here.parent / "dados",
            here.parent.parent / "dados",
        ]
    )
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError(
        "Não achei ravenstack_accounts.csv. Defina o widget data_path "
        "para a pasta dados/ no Workspace ou em um Volume."
    )


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "ravenstack")
data_path = resolve_data_path(widget_or_default("data_path", ""))
print(f"catalog={catalog_name} schema={schema_name}")
print(f"data_path={data_path}")

In [ ]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [ ]:
from pyspark.sql import functions as F

loaded = {}
for entity_name, contract in EXPECTED.items():
    csv_path = str(Path(data_path) / f"{entity_name}.csv")
    table_name = f"{catalog_name}.{schema_name}.{entity_name}"
    sdf = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .option("nullValue", "")
        .option("mode", "PERMISSIVE")
        .csv(csv_path)
    )
    missing = [col for col in contract["columns"] if col not in sdf.columns]
    extra = [col for col in sdf.columns if col not in contract["columns"]]
    if missing or extra:
        raise ValueError(
            f"{entity_name}: colunas fora do CSV. missing={missing} extra={extra}"
        )
    sdf = sdf.select(*contract["columns"])
    for column in sdf.columns:
        if column in BOOL_COLUMNS:
            sdf = sdf.withColumn(column, F.col(column).cast("boolean"))
        elif column in DATE_COLUMNS:
            sdf = sdf.withColumn(column, F.to_date(F.col(column).cast("string")))
        elif column in TIMESTAMP_COLUMNS:
            sdf = sdf.withColumn(column, F.to_timestamp(F.col(column).cast("string")))
    row_count = sdf.count()
    if row_count != contract["rows"]:
        raise ValueError(
            f"{entity_name}: esperado {contract['rows']} linhas, lido {row_count}"
        )
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    loaded[entity_name] = {"rows": row_count, "cols": sdf.columns}
    print(f"{table_name} rows={row_count}")

loaded

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

In [ ]:
for entity_name in EXPECTED:
    print(f"\n===== {entity_name} =====")
    display(spark.table(f"{catalog_name}.{schema_name}.{entity_name}").limit(5))